[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kkaitechnology-coder/free-python-for-development/blob/main/lesson-6-3-web-scraping-and-automation.ipynb)

# Lesson 6.3 · Web Scraping & Automation

**Free Python For Development** · Topic 6: Advanced Python & Real-World Projects

_BeautifulSoup, Selenium, Scrapy._

▶ **Watch this lesson with the AI voice tutor:** [open it in OratorLab](https://oratorlab.in/course-player.html?courseId=python-for-development&subtopicId=subtopic-6-3)

**How to use this notebook:** run each code cell with **Shift + Enter**, or use **Runtime → Run all**. Then change the code and run it again — experimenting is the fastest way to learn.

**Setup** — install the libraries this lesson uses (takes a few seconds):

In [ ]:
!pip install -q requests beautifulsoup4

## Introduction & Core Concepts of Web Scraping & Automation

Imagine you want to collect all the latest news headlines from a website, or monitor price changes on your favorite online store. Doing this manually would take hours. With Python you can automate these repetitive tasks using **web scraping** and automation tools.

Web scraping is like sending a robot to a website to fetch information for you. The robot downloads the page, reads the HTML, and extracts the pieces you care about — headlines, prices, product names, links, and more.

> 💡 **Key insight:** The three most common tools form a clear progression: **BeautifulSoup** for static HTML, **Selenium** for interactive/JavaScript-heavy pages, and **Scrapy** for large-scale, structured crawling projects.

### The Basic Pipeline

- **Request** — download the page (requests, httpx, or a browser).
- **Parse** — turn the HTML into a searchable tree (BeautifulSoup, lxml, parsel).
- **Extract** — select the elements you need with CSS selectors or XPath.
- **Store / act** — save to CSV, database, or trigger further automation.

> 💡 **Mental model:** Think of a website as a messy filing cabinet. Scraping tools give you a map and a pair of hands so you can pull out exactly the folders you need without reading every page yourself.

## BeautifulSoup & Selenium — From Static to Interactive

**BeautifulSoup** helps you navigate and extract data from HTML pages. It is ideal when the content you need is already present in the initial HTML response.

In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://news.ycombinator.com"
resp = requests.get(url, timeout=10)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, "html.parser")

# Extract top story titles
titles = []
for a in soup.select("span.titleline > a")[:5]:
    titles.append(a.get_text(strip=True))

for i, t in enumerate(titles, 1):
    print(f"{i}. {t}")

**BeautifulSoup strengths**

- Simple API, fast for static pages.
- CSS selectors and tag navigation.
- Pairs perfectly with the requests library.
- Lightweight — no browser needed.

When a site is heavily driven by JavaScript, requires login, or needs button clicks and form fills, you need a real browser. That is where **Selenium** shines — it drives Chrome, Firefox or Edge programmatically.

Selenium drives a real Chrome browser, so run this one **on your computer** (`pip install selenium`, with Chrome installed). `https://example.com/login` is a placeholder — point it at a login page you are allowed to automate.

```python
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time

options = Options()
options.add_argument("--headless=new")  # run without UI

driver = webdriver.Chrome(options=options)
driver.get("https://example.com/login")

driver.find_element(By.NAME, "username").send_keys("demo")
driver.find_element(By.NAME, "password").send_keys("secret")
driver.find_element(By.CSS_SELECTOR, "button[type=submit]").click()

time.sleep(2)  # wait for redirect / JS
print("Title after login:", driver.title)
driver.quit()
```

**When to choose Selenium**

- Content appears only after JavaScript runs.
- You must log in or click through multi-step flows.
- You need to scroll, hover, or handle modals.
- Trade-off: heavier and slower than pure HTTP scraping.

## Scrapy & Production Best Practices

For larger, more complex scraping projects, **Scrapy** is the industry-standard framework. Think of it as an assembly line: it efficiently schedules requests, handles concurrency, follows links, and pipelines the extracted data into storage.

**Scrapy advantages**

Built-in concurrency, robots.txt respect, auto-throttling, item pipelines, export to JSON/CSV/databases, powerful selectors, and middleware for headers, proxies, and retries.

**Typical use cases**

Crawling entire product catalogs, news archives, directory sites, or any multi-page structured data collection at scale.

### Minimal Scrapy Spider Sketch

In [ ]:
!pip install -q scrapy

In [ ]:
%%writefile quotes_spider.py
# quotes_spider.py (inside a Scrapy project)
import scrapy

class QuotesSpider(scrapy.Spider):
    name = "quotes"
    start_urls = ["https://quotes.toscrape.com"]

    def parse(self, response):
        for q in response.css("div.quote"):
            yield {
                "text": q.css("span.text::text").get(),
                "author": q.css("small.author::text").get(),
                "tags": q.css("div.tags a.tag::text").getall(),
            }

        next_page = response.css("li.next a::attr(href)").get()
        if next_page:
            yield response.follow(next_page, callback=self.parse)

In [ ]:
# Inside a Scrapy project you would run: scrapy crawl quotes -o quotes.json
# For a single file, runspider does the same:
!scrapy runspider quotes_spider.py -O quotes.json -s LOG_LEVEL=WARNING

In [ ]:
import json

quotes = json.load(open("quotes.json", encoding="utf-8"))
print(len(quotes), "quotes scraped")
for q in quotes[:3]:
    print(f'- {q["author"]}: {q["text"][:70]}')

**Production checklist**

- Respect `robots.txt` and site terms of service.
- Set a reasonable crawl rate / use AutoThrottle.
- Rotate user-agents and consider proxies when needed.
- Handle errors and retries gracefully.
- Cache responses during development.
- Never scrape personal data without clear legal basis.

> 💡 **Big takeaway:** BeautifulSoup is perfect for static pages, Selenium controls a real browser for dynamic sites, and Scrapy scales to industrial-strength crawling. Together they turn the web into a programmable data source — just remember to scrape responsibly and ethically.

---

✅ **Lesson complete!** Next up: **Lesson 6.4 · Package Publishing & Deployment** — [watch it in OratorLab](https://oratorlab.in/course-player.html?courseId=python-for-development&subtopicId=subtopic-6-4) or open `lesson-6-4-package-publishing-and-deployment.ipynb`.

Learn the full **Free Python For Development** course for free at [oratorlab.in](https://oratorlab.in/courses/python-for-development).